# UCF101 Dataset embeddings generator script

This notebook is for using the CLIP model to generate the embeddings/features for each frame of the videos in the UCF101 dataset. Google Colab is used as it provides easy access to a powerful (T4) GPU. This massively reduces the time to perform inference on all the frames for the videos that are being processed in this notebook.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers tqdm

## Define the paths for how to process the data

You must specify the paths to find the .zip file containing the dataset folder, where to save the output, and more...


In [3]:
import os
from pathlib import Path

# Path to the zip with the dataset folder inside
DRIVE_ZIP_PATH = "/content/drive/My Drive/Dissertation/data/ucf101_subset.zip"

# Path to the current session's (local) data directory
LOCAL_DATA_DIR = Path("/content") / "data"

# Name of the directory of [videos + csv]
DATASET_DIR_NAME = "ucf101_subset"

# Path to directory of the dataset
LOCAL_DATASET_ROOT = LOCAL_DATA_DIR / DATASET_DIR_NAME

# Name of the CSV file containing information on the videos [csv]
DATASET_CSV_FILE_NAME = "test.csv"

# Path to the dataset's CSV file containing information about videos
LOCAL_CSV_PATH = LOCAL_DATASET_ROOT / DATASET_CSV_FILE_NAME

# Location where the embeddings will be saved
GDRIVE_OUTPUT_DIR = Path("/content/drive/My Drive/Dissertation/data/ucf101_embeddings/")

## Unzip the dataset folder


In [4]:
print("Unzipping video data... (this may take a few minutes)")
!unzip -oq "{DRIVE_ZIP_PATH}" -d "{str(LOCAL_DATA_DIR)}"
print("Data unzipped successfully.")

Unzipping video data... (this may take a few minutes)
replace /content/data/ucf101_subset/test/ApplyEyeMakeup/v_ApplyEyeMakeup_g01_c02.avi? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Data unzipped successfully.


In [5]:
# Check that the data has been unzipped correctly
!ls "{str(LOCAL_DATA_DIR)}"
print()
!ls "{str(LOCAL_DATA_DIR / 'ucf101_subset')}" | head -n 5
print()
!ls "{str(LOCAL_DATA_DIR / 'ucf101_subset' / 'test')}" | head -n 5
print()
!ls "{str(LOCAL_DATA_DIR / 'ucf101_subset' / 'test' / 'Swing')}" | head -n 5

ucf101_subset

test
test.csv

ApplyEyeMakeup
ApplyLipstick
Archery
BabyCrawling
BalanceBeam

v_Swing_g01_c02.avi
v_Swing_g04_c03.avi
v_Swing_g04_c04.avi
v_Swing_g04_c05.avi
v_Swing_g11_c04.avi


## Load the model in for inference


In [6]:
import torch
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import cv2
from tqdm import tqdm
import time
import numpy as np

# Set up model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/clip-vit-base-patch32"
BATCH_SIZE = 64

print(f"Loading CLIP model '{MODEL_ID}' onto {DEVICE}...")
model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE)
processor = CLIPProcessor.from_pretrained(MODEL_ID)
print("Model loaded.")

Loading CLIP model 'openai/clip-vit-base-patch32' onto cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model loaded.


## Load in the dataset information file (CSV)


In [7]:
df = pd.read_csv(LOCAL_CSV_PATH)
df.head()

,clip_name,clip_path,label,frame_count,duration_sec,fps,width,height,resolution
0,v_Swing_g21_c02,/test/Swing/v_Swing_g21_c02.avi,Swing,168,6.72,25.0,320,240,320x240
1,v_Swing_g21_c06,/test/Swing/v_Swing_g21_c06.avi,Swing,201,8.04,25.0,320,240,320x240
2,v_Swing_g20_c05,/test/Swing/v_Swing_g20_c05.avi,Swing,151,6.04,25.0,320,240,320x240
3,v_Swing_g04_c03,/test/Swing/v_Swing_g04_c03.avi,Swing,168,6.72,25.0,320,240,320x240
4,v_Swing_g19_c03,/test/Swing/v_Swing_g19_c03.avi,Swing,276,11.04,25.0,320,240,320x240


In [8]:
def extract_all_frames(video_path: str):
    """
    Opens a video file and extracts all frames into a list of RGB images (as required for CLIP).
    """
    frames = []
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Warning: Could not open video: {video_path}")
        return None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

## Test run of running inference on one video


In [ ]:
# Estimate Time to process videos
print("Starting Time Estimation (1 Video)")
start_time = time.time()

# Get one video to test
test_row = df.iloc[0]
test_video_path = LOCAL_DATASET_ROOT / test_row["clip_path"].lstrip("/")

test_frames = extract_all_frames(str(test_video_path))

if test_frames:
    with torch.no_grad():
        for i in range(0, len(test_frames), BATCH_SIZE):
            batch = test_frames[i : i + BATCH_SIZE]
            # The processor handles resizing (to 224x224) and normalization
            inputs = processor(
                text=None, images=batch, return_tensors="pt", padding=True
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            model.get_image_features(**inputs)  # Run inference

end_time = time.time()
time_per_video = end_time - start_time
total_videos = len(df)
estimated_total_time_min = (time_per_video * total_videos) / 60

if test_frames:
    print(
        f"Time for 1 video ({len(test_frames)} frames): {time_per_video:.2f} seconds."
    )
else:
    print(
        f"Warning: Failed to process test video {test_video_path}, cannot estimate time per video."
    )
    print(f"Estimated total time for {total_videos} videos: N/A minutes.")

print("Time Estimation Complete")

Starting Time Estimation (1 Video)
Time for 1 video (168 frames): 45.95 seconds.
Time Estimation Complete


## Run inference on all videos and save embeddings to output directory

This may take a long time (as expected).

To make this run as fast as possible, make sure to switch the google colab runtime to one with a GPU.

'tqdm' provides a live progress bar below.


In [ ]:
print(f"Starting Full Embedding Pre-computation ({total_videos} videos)")

for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing videos"):
    video_relative_path = Path(row["clip_path"].lstrip("/"))
    # Full Path to the video
    full_video_path = LOCAL_DATASET_ROOT / video_relative_path

    embedding_relative_path = video_relative_path.with_suffix(".pt")
    # Path to output location for this video
    output_path = GDRIVE_OUTPUT_DIR / embedding_relative_path

    # Convert clip_path in DataFrame to the new embedding relative path format
    df.at[index, "clip_path"] = str(embedding_relative_path)

    # This makes the script resumable if it crashes
    if output_path.exists():
        continue

    # Create directories if they don't exist
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Get the video's frames
    frames = extract_all_frames(str(full_video_path))

    if not frames:
        print(f"Warning: Failed to read frames from {full_video_path}, skipping.")
        continue

    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(frames), BATCH_SIZE):
            batch_frames = frames[i : i + BATCH_SIZE]

            # The processor handles resizing (to 224x224) and normalization
            inputs = processor(
                text=None, images=batch_frames, return_tensors="pt", padding=True
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            image_features = model.get_image_features(**inputs)  # Run inference
            all_embeddings.append(image_features.cpu())

    if not all_embeddings:
        print(f"Warning: No embeddings generated for {full_video_path}, skipping.")
        continue

    # Combine all embeddings into one tensor
    final_embeddings = torch.cat(all_embeddings, dim=0)
    torch.save(final_embeddings, output_path)

updated_csv_path = GDRIVE_OUTPUT_DIR / DATASET_CSV_FILE_NAME
df.to_csv(updated_csv_path, index=False)

print("\nPre-computation Finished!")

Starting Full Embedding Pre-computation (1723 videos)


Processing videos: 100%|██████████| 1723/1723 [00:01<00:00, 924.28it/s]


Pre-computation Finished!


## The new video embeddings dataset can be found in your output directory.
